# Organized execution copy

The exact executed predecessor with historical outputs is in `archive/revision_executed/`. This execution copy has cleared outputs. No new GPU experiment was run during packaging. Install, restart, then run the configuration cell and remaining cells in order. Edit DATA_ROOT and RUN_DIR consistently across notebooks; use a new RUN_DIR for a new reproduction and the same one to resume it. See `docs/RUN_ORDER.md` and `docs/DATA_SETUP.md` for required files. The conversion/model/scoring calls and experimental settings are retained.

Run on a different day in a fresh T4 session. Scoring models are intentionally excluded from timing setup.


In [ ]:
# Cell 1 — INSTALL
# After completion: Runtime → Restart session

import os
import sys
import subprocess

print("Python:", sys.version, flush=True)

OPENVOICE_REPO = "/content/OpenVoice"
OPENVOICE_COMMIT = "74a1d147b17a8c3092dd5430504bd83ef6c7eb23"

SEED_REPO = "/content/seed-vc"
SEED_COMMIT = "51383efd921027683c89e5348211d93ff12ac2a8"

repositories = [
    (
        OPENVOICE_REPO,
        "https://github.com/myshell-ai/OpenVoice.git",
        OPENVOICE_COMMIT,
    ),
    (
        SEED_REPO,
        "https://github.com/Plachtaa/seed-vc.git",
        SEED_COMMIT,
    ),
]

for repo, url, commit in repositories:
    print(f"\nPreparing {repo}...", flush=True)

    if not os.path.isdir(repo):
        subprocess.run(
            ["git", "clone", "--progress", url, repo],
            check=True,
        )

    subprocess.run(
        ["git", "checkout", commit],
        cwd=repo,
        check=True,
    )

    print(f"Repository ready: {repo}", flush=True)

pkgs = [
    # OpenVoice
    "wavmark==0.0.3",
    "eng_to_ipa==0.0.2",
    "inflect==7.0.0",
    "unidecode==1.3.7",
    "pypinyin==0.50.0",
    "cn2an==0.5.22",
    "jieba==0.42.1",
    "langid==1.1.6",

    # Seed-VC
    "numpy==1.26.4",
    "scipy==1.13.1",
    "librosa==0.10.2",
    "transformers==4.46.3",
    "huggingface-hub>=0.28.1",
    "munch==4.0.0",
    "einops==0.8.0",
    "descript-audio-codec==1.0.0",
    "resemblyzer",
    "soundfile==0.12.1",
    "modelscope==1.18.1",
    "funasr==1.1.5",
    "hydra-core==1.3.2",
    "python-dotenv",
    "pyyaml",

    # Scoring
    "speechbrain==1.0.3",
    "jiwer==3.0.3",
    "accelerate",
]

print("\nInstalling packages...", flush=True)

process = subprocess.Popen(
    [
        sys.executable,
        "-u",
        "-m",
        "pip",
        "install",
        "--progress-bar",
        "off",
    ] + pkgs,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

for line in process.stdout:
    print(line, end="", flush=True)

return_code = process.wait()

if return_code != 0:
    raise RuntimeError(
        f"Package installation failed: {return_code}"
    )

print(
    "\n✅ Installed. NOW: Runtime → Restart session, "
    "then run Cell 2.",
    flush=True,
)

In [ ]:
# Runtime configuration — edit these two lines consistently in all notebooks.
import os
DATA_ROOT = "/content/drive/MyDrive/IJEEE_VC_data"
RUN_DIR = "/content/drive/MyDrive/IJEEE_VC_data/reproduction_runs/review_run_01"
# DATA_ROOT contains audio/models. RUN_DIR is a new output folder, never results/.
os.environ["IJEEE_DATA_ROOT"] = DATA_ROOT
os.environ["IJEEE_RUN_DIR"] = RUN_DIR


In [ ]:
# Cell 2 — PATHS AND SETTINGS
import os, sys, glob, json, time, random, shutil, tempfile, re, string, unicodedata, subprocess
os.environ["USE_TF"] = "0"; os.environ["TRANSFORMERS_NO_TF"] = "1"; os.environ["USE_FLAX"] = "0"
from google.colab import drive
drive.mount("/content/drive")
import numpy as np, pandas as pd, torch

BASE_DIR = os.environ.get("IJEEE_DATA_ROOT", "/content/drive/MyDrive/IJEEE_VC_data")
OUT = os.environ.get("IJEEE_RUN_DIR", os.path.join(BASE_DIR, "reproduction_runs", "review_run_01")); os.makedirs(OUT, exist_ok=True)

TUNING_FILES = [os.path.join(BASE_DIR, "selected_hybrid_data", f"joe_source_{i:02d}.wav") for i in (1, 2, 3)]
TUNING_TEXT = ["The monkey was planning to throw the net like the fishermen did.",
               "Gaze out to sea, picnic on benches and watch fishermen hook herring.",
               "We all have our own methods of dealing with money, but here are a couple of saving tricks that will grow your money."]
TARGET_REF = os.path.join(BASE_DIR, "selected_hybrid_data", "kathleen_reference.wav")
# EDIT if the two new conditioning references are stored elsewhere on Drive:
EXTRA_REFS = [os.path.join(BASE_DIR, "selected_hybrid_data", f"kathleen_conditioning_ref_{i:02d}.wav") for i in (2, 3)]
EVAL_CLIPS = sorted(glob.glob(os.path.join(BASE_DIR, "kathleen_independent_refs", "kathleen_eval_*.wav")))
JOE_ARCHIVE = os.path.join(BASE_DIR, "joe.tar.gz")
HOLDOUT10_META = os.path.join(BASE_DIR, "final_holdout_data", "holdout_metadata.csv")
HOLDOUT40_DIR = os.path.join(BASE_DIR, "final_holdout_data_40"); os.makedirs(HOLDOUT40_DIR, exist_ok=True)
HOLDOUT40_META = os.path.join(HOLDOUT40_DIR, "holdout40_metadata.csv")
EXISTING = os.path.join(BASE_DIR, "final_hybrid_comparison")
OV_CONVERTER = os.path.join(BASE_DIR, "models", "OpenVoiceV2", "converter")
N_HOLDOUT = 40
TUNING_IDS = {"0000000160", "0000000229", "0000000068"}

for p in TUNING_FILES + [TARGET_REF, JOE_ARCHIVE, HOLDOUT10_META, os.path.join(OV_CONVERTER, "checkpoint.pth")]:
    assert os.path.isfile(p), f"❌ missing: {p}"
for p in EXTRA_REFS:
    print("extra reference", os.path.basename(p), "found" if os.path.isfile(p) else "❌ NOT FOUND — fix the EDIT line")
print("Independent Kathleen evaluation clips:", len(EVAL_CLIPS))
assert len(EVAL_CLIPS) == 10, "❌ kathleen_eval_*.wav clips not found"
assert torch.cuda.is_available(), "❌ select a T4 GPU runtime"
print("GPU:", torch.cuda.get_device_name(0))

def set_all_seeds(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
def append_row(csv, row):
    # Schema validation and atomic replacement happen outside the measured span.
    # Existing malformed files are rejected, never silently repaired or skipped.
    import csv as csv_module
    import io
    import tempfile
    target = os.path.abspath(csv)
    parent = os.path.dirname(target)
    incoming = list(row)
    old_rows = []
    if os.path.exists(target) and os.path.getsize(target):
        with open(target, newline="", encoding="utf-8") as handle:
            reader = csv_module.reader(handle, strict=True)
            columns = next(reader)
            if len(columns) != len(set(columns)):
                raise ValueError(f"Duplicate CSV columns: {target}")
            if set(columns) != set(incoming):
                raise ValueError(f"CSV schema differs: {target}; preserve the file and inspect it before resuming")
            for number, values in enumerate(reader, start=2):
                if len(values) != len(columns):
                    raise ValueError(f"Malformed CSV row {number}: {target}; historical recovery must be performed separately")
                old_rows.append(values)
    else:
        columns = incoming
    # pandas encodes floats, NaN and transcript quoting as in the original code.
    encoded = pd.DataFrame([row], columns=columns).to_csv(index=False, header=False)
    new_values = list(csv_module.reader(io.StringIO(encoded), strict=True))
    if len(new_values) != 1 or len(new_values[0]) != len(columns):
        raise ValueError("Expected exactly one valid result row")
    descriptor, temporary = tempfile.mkstemp(prefix=".result_", suffix=".csv.tmp", dir=parent)
    try:
        with os.fdopen(descriptor, "w", newline="", encoding="utf-8") as handle:
            writer = csv_module.writer(handle)
            writer.writerow(columns)
            writer.writerows(old_rows)
            writer.writerow(new_values[0])
            handle.flush()
            os.fsync(handle.fileno())
        os.replace(temporary, target)
    finally:
        if os.path.exists(temporary):
            os.remove(temporary)
def done_keys(csv, cols):
    if not os.path.exists(csv): return set()
    return set(map(tuple, pd.read_csv(csv)[cols].astype(str).values))
print("✅ Paths ready")

In [ ]:
# Cell 3 — EXTEND THE HOLDOUT SET TO 40 SENTENCES

from pathlib import Path
import tarfile

LOCAL_JOE = "/content/final_comparison_dataset/joe"
os.makedirs(LOCAL_JOE, exist_ok=True)

if not glob.glob(
    os.path.join(LOCAL_JOE, "**", "*.webm"),
    recursive=True,
):
    with tarfile.open(JOE_ARCHIVE, "r:gz") as archive:
        archive.extractall(LOCAL_JOE, filter="data")

files = []

for extension in [
    "*.webm", "*.wav", "*.flac", "*.mp3",
    "*.ogg", "*.opus", "*.m4a",
]:
    files.extend(
        glob.glob(
            os.path.join(LOCAL_JOE, "**", extension),
            recursive=True,
        )
    )

files = sorted(set(files))
print("Joe files:", len(files))

candidates = [
    path
    for path in files
    if Path(path).stem not in TUNING_IDS
]

random.Random(20260822).shuffle(candidates)


def dur(path):
    result = subprocess.run(
        [
            "ffprobe",
            "-v", "error",
            "-show_entries", "format=duration",
            "-of", "default=noprint_wrappers=1:nokey=1",
            path,
        ],
        capture_output=True,
        text=True,
    )

    try:
        return float(result.stdout.strip())
    except ValueError:
        return None


def txt(path):
    transcript_path = Path(path).with_suffix(".txt")

    if not transcript_path.is_file():
        return None

    transcript = " ".join(
        transcript_path.read_text(
            encoding="utf-8",
            errors="ignore",
        ).split()
    )

    return transcript or None


records = []

for path in candidates:
    if len(records) == N_HOLDOUT:
        break

    duration = dur(path)

    if duration is None or not (4.0 < duration < 7.5):
        continue

    transcript = txt(path)

    if transcript is None:
        continue

    sample_number = len(records) + 1

    output_path = os.path.join(
        HOLDOUT40_DIR,
        f"joe_holdout_{sample_number:02d}.wav",
    )

    if not os.path.isfile(output_path):
        subprocess.run(
            [
                "ffmpeg",
                "-y",
                "-v", "error",
                "-i", path,
                "-ac", "1",
                "-ar", "22050",
                "-c:a", "pcm_s16le",
                output_path,
            ],
            check=True,
        )

    records.append(
        {
            "sample": sample_number,
            "original_id": Path(path).stem,
            "duration_seconds": duration,
            "transcript": transcript,
            "prepared_wav": output_path,
        }
    )

    print(
        f"Prepared {sample_number}/{N_HOLDOUT}: "
        f"{Path(path).stem}",
        flush=True,
    )

assert len(records) == N_HOLDOUT, (
    f"❌ Expected {N_HOLDOUT} sentences, found {len(records)}"
)

H40 = pd.DataFrame(records)

# Preserve leading zeros in recording IDs.
old10 = pd.read_csv(
    HOLDOUT10_META,
    dtype={"original_id": str},
)

same = (
    H40["original_id"].astype(str).head(10).tolist()
    == old10["original_id"].astype(str).tolist()
)

print("First 10 identical to the paper's holdout:", same)

assert same, (
    "❌ The extended selection does not start with "
    "the original 10 — stop and report"
)

H40.to_csv(HOLDOUT40_META, index=False)

print(
    H40[
        ["sample", "original_id", "duration_seconds"]
    ].tail(5)
)

print("✅ 40 holdout sentences ready")

In [ ]:
# Cell 4 — LOAD MODELS, WARM-UP, CONVERSION FUNCTIONS (from Untitled15)
from types import SimpleNamespace
sys.path.insert(0, "/content/OpenVoice")
from openvoice.api import ToneColorConverter
tone_color_converter = ToneColorConverter(os.path.join(OV_CONVERTER, "config.json"), device="cuda:0")
tone_color_converter.load_ckpt(os.path.join(OV_CONVERTER, "checkpoint.pth"))
sys.path.insert(0, "/content/seed-vc"); os.chdir("/content/seed-vc")
import inference as seed_inference
def seed_args(src, ref, out, steps):
    return SimpleNamespace(source=src, target=ref, output=out, diffusion_steps=steps, length_adjust=1.0,
                           inference_cfg_rate=0.7, f0_condition=False, auto_f0_adjust=False, semi_tone_shift=0,
                           checkpoint=None, config=None, fp16=True)
seed_model_bundle = seed_inference.load_models(seed_args(None, None, None, 10))
seed_inference.load_models = lambda args: seed_model_bundle       # never reload inside main()

SE_CACHE = {}
def get_se(path):                     # cached, outside the timed span (as in the paper)
    if path not in SE_CACHE: SE_CACHE[path] = tone_color_converter.extract_se(path)
    return SE_CACHE[path]
def run_openvoice(src, ref, tau, out_path, seed):
    src_se, tgt_se = get_se(src), get_se(ref)
    set_all_seeds(seed); torch.cuda.synchronize(); t0 = time.perf_counter()
    tone_color_converter.convert(audio_src_path=src, src_se=src_se, tgt_se=tgt_se, output_path=out_path, tau=tau, message="@MyShell")
    torch.cuda.synchronize(); return time.perf_counter() - t0
os.makedirs("/content/seed_tmp", exist_ok=True)
def run_seedvc(src, ref, steps, out_path, seed):
    work = tempfile.mkdtemp(dir="/content/seed_tmp")
    set_all_seeds(seed); torch.cuda.synchronize(); t0 = time.perf_counter()
    seed_inference.main(seed_args(src, ref, work, steps))
    torch.cuda.synchronize(); el = time.perf_counter() - t0
    produced = sorted(glob.glob(os.path.join(work, "*.wav")), key=os.path.getmtime)
    assert produced, "❌ Seed-VC produced no file"
    shutil.move(produced[-1], out_path); shutil.rmtree(work, ignore_errors=True)
    return el
def run_cascade_e2e(src, ref, tau, steps, out_path, ov_seed, sv_seed):
    # one end-to-end timed run: both stages and the intermediate file inside one timed span
    inter = out_path.replace(".wav", "_e2e_ov.wav")
    torch.cuda.synchronize(); t0 = time.perf_counter()
    run_openvoice(src, ref, tau, inter, ov_seed); run_seedvc(inter, ref, steps, out_path, sv_seed)
    torch.cuda.synchronize(); return time.perf_counter() - t0

WARM = "/content/warmup"; os.makedirs(WARM, exist_ok=True)
for _ in range(2):
    run_openvoice(TUNING_FILES[0], TARGET_REF, 0.5, f"{WARM}/ov.wav", 2026)
    run_seedvc(TUNING_FILES[0], TARGET_REF, 10, f"{WARM}/sv.wav", 2028)
ENV = dict(gpu=torch.cuda.get_device_name(0), torch=torch.__version__, openvoice_commit="74a1d147b17a8c3092dd5430504bd83ef6c7eb23",
           seedvc_commit="51383efd921027683c89e5348211d93ff12ac2a8", seedvc_fp16=True, openvoice_precision="fp32",
           batch_size=1, warmups=2, timed_repeats=3, cuda_synchronize=True, openvoice_embeddings_cached=True)
json.dump(ENV, open(os.path.join(OUT, "environment.json"), "w"), indent=1)
print("✅ Models loaded and warmed up"); print(ENV)

In [ ]:
# Cell 11 — SECOND-SESSION TIMING (timing only, no scoring)
H40 = pd.read_csv(HOLDOUT40_META)
csv = os.path.join(OUT, "E11_session_timing.csv")
# A session is a runtime-local identifier; rerunning resumes its completed keys.
import uuid
if "TIMING_SESSION_ID" not in globals():
    TIMING_SESSION_ID = time.strftime("%Y-%m-%d_%H%M%S") + "_" + uuid.uuid4().hex[:8]
session = TIMING_SESSION_ID
completed = done_keys(csv, ["session", "system", "sample"])
d11 = os.path.join("/content", "E11_audio"); os.makedirs(d11, exist_ok=True)
for name, kind, tau, steps in [("OpenVoice V2 (tau 0.5)", "ov", 0.5, None), ("Seed-VC (10 steps)", "sv", None, 10),
                               ("Reference cascade (tau 0.3, 20)", "casc", 0.3, 20), ("Selected cascade (tau 0.5, 10)", "casc", 0.5, 10)]:
    for _, r in H40.head(10).iterrows():
        i = int(r["sample"])
        if (session, name, str(i)) in completed: continue
        out = os.path.join(d11, f"x_s{i}.wav"); inter = out.replace(".wav", "_ov.wav")
        t_ov = float(np.mean([run_openvoice(r.prepared_wav, TARGET_REF, tau, inter, 202700 + i) for _ in range(3)])) if kind != "sv" else 0.0
        t_sv = float(np.mean([run_seedvc(inter if kind == "casc" else r.prepared_wav, TARGET_REF, steps, out, 202900 + i) for _ in range(3)])) if kind != "ov" else 0.0
        e2e = run_cascade_e2e(r.prepared_wav, TARGET_REF, tau, steps, out, 202700 + i, 202900 + i) if kind == "casc" else np.nan
        append_row(csv, dict(session=session, gpu=torch.cuda.get_device_name(0), system=name, sample=i,
                             t_openvoice=t_ov, t_seedvc=t_sv, latency=t_ov + t_sv, latency_e2e=e2e))
print(pd.read_csv(csv).groupby(["session", "system"])[["latency", "latency_e2e"]].mean().round(3))

In [ ]:
timing = pd.read_csv(
    os.path.join(OUT, "E11_session_timing.csv")
)

print(
    timing.groupby(["session", "system"])
    .agg(rows=("sample", "size"), samples=("sample", "nunique"))
)

In [ ]:

# Cell 13 — PACK RESULTS (CSVs + environment) AND DOWNLOAD
import zipfile
from google.colab import files
zp = "/content/IJEEE_revision_results_final.zip"
with zipfile.ZipFile(zp, "w") as z:
    for f in glob.glob(os.path.join(OUT, "*.csv")) + glob.glob(os.path.join(OUT, "*.json")) + [HOLDOUT40_META]:
        z.write(f, os.path.basename(f))
print("Files in zip:", len(zipfile.ZipFile(zp).namelist()))
files.download(zp)